# Notebook 3 — Automated Question Generation

**Goal:** convert the scraped articles from Notebook 2 into machine-gradable
multiple-choice questions, serialized as strict JSON in `data/questions/`.

## Why synthetic test generation?

Auditing a model's knowledge traditionally requires human-written questions —
slow, expensive, and impossible to refresh daily. The alternative: use an LLM
as an *item writer*. Given a source document, it drafts fact-based questions
whose answers are contained in (and only in) that document. Because we hold
the source text, we can **verify each generated item mechanically** before
trusting it:

1. **Structural validity** — exactly 4 options, correct index in range
   (enforced by the Pydantic schema + strict JSON decoding).
2. **Groundedness** — the model must quote an `evidence_excerpt`; we check it
   actually appears in the article. An item whose "evidence" was hallucinated
   is discarded. *Validate the test generator just like the test taker.*

The separation of roles is the key methodological idea: the **generator**
model reads the article; the **evaluated** model (Notebook 4) never sees it.

Generation runs through the same `toolkit.providers.openai_provider` used in
Notebook 1 — Responses API, structured outputs, retry — but with
`use_web_search=False` (the article is the only source we want) and a custom
`text_format` schema.


In [ ]:
import os
from pathlib import Path

# Notebooks live in notebooks/; run from the repo root so relative paths like
# ./data behave identically in VS Code and classic Jupyter. The local
# `toolkit` package is installed (editable) in the .venv, so no sys.path hacks.
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
os.chdir(repo_root)
print(f"Working directory: {Path.cwd()}")

import json

import pandas as pd
from pydantic import BaseModel, Field

from toolkit.config import ARTICLES_DIR, DEFAULT_LLM, QUESTIONS_DIR
from toolkit.providers import openai_provider

articles_dir = Path(ARTICLES_DIR)
questions_dir = Path(QUESTIONS_DIR)
questions_dir.mkdir(parents=True, exist_ok=True)

article_paths = sorted(articles_dir.glob("article_*.txt"))
if not article_paths:
    raise FileNotFoundError(
        "No articles found in data/articles/ — run Notebook 2 first."
    )
print(f"Found {len(article_paths)} articles: {[p.name for p in article_paths]}")


## 1. Question schema

`MCQuestion` defines a single item; `MCQuestionSet` wraps a list so one
structured call returns several questions per article.


In [ ]:
class MCQuestion(BaseModel):
    question: str = Field(description="Fact-based multiple choice question based on the text.")
    options: list[str] = Field(description="A list containing exactly 4 options.")
    correct_index: int = Field(description="The index (0-3) of the correct answer option.")
    evidence_excerpt: str = Field(description="The exact text snippet that supports this correct answer.")


class MCQuestionSet(BaseModel):
    questions: list[MCQuestion] = Field(description="The list of generated questions.")


## 2. Generation + mechanical validation

The prompt pins down the failure modes we care about: self-contained
questions (answerable without seeing the article *if you know the facts*),
plausible-but-wrong distractors, and a verbatim evidence quote. Articles are
truncated to ~8,000 characters to keep token costs predictable.


In [ ]:
GENERATOR_SYSTEM = (
    "You write factual multiple-choice quiz questions from news articles. "
    "Rules: every question must be answerable from the article alone; provide "
    "exactly 4 answer options with exactly one correct; distractors must be "
    "plausible but clearly wrong given the article; the question must be "
    "self-contained (name the entities involved — never say 'according to the "
    "article'); evidence_excerpt must be copied VERBATIM from the article text."
)

MAX_ARTICLE_CHARS = 8000


def generate_questions(article_text, n=3):
    parsed, _raw = openai_provider.run_parsed(
        DEFAULT_LLM,
        GENERATOR_SYSTEM,
        (
            f"Generate exactly {n} fact-based multiple-choice questions "
            f"from this article:\n\n{article_text[:MAX_ARTICLE_CHARS]}"
        ),
        use_web_search=False,  # the article is the only source we want
        text_format=MCQuestionSet,
    )
    return parsed


def normalize_ws(s):
    return " ".join(s.split()).lower()


def validate_question(q, article_text):
    problems = []
    if len(q.options) != 4:
        problems.append(f"expected 4 options, got {len(q.options)}")
    if not 0 <= q.correct_index <= 3:
        problems.append(f"correct_index {q.correct_index} out of range")
    if normalize_ws(q.evidence_excerpt) not in normalize_ws(article_text):
        problems.append("evidence_excerpt not found verbatim in article")
    return problems


In [ ]:
for path in article_paths:
    article_text = path.read_text(encoding="utf-8")
    idx = path.stem.split("_")[-1]
    out_path = questions_dir / f"question_{idx}.json"
    print(f"\n{path.name} ({len(article_text):,} chars)")
    try:
        question_set = generate_questions(article_text, n=3)
    except Exception as e:
        print(f"  ! generation failed: {e}")
        continue

    valid = []
    for q in question_set.questions:
        problems = validate_question(q, article_text)
        if problems:
            print(f"  DROPPED: {q.question[:60]}...  ({'; '.join(problems)})")
        else:
            valid.append(q)

    payload = {
        "source_article": path.name,
        "questions": [q.model_dump() for q in valid],
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    print(f"  kept {len(valid)}/{len(question_set.questions)} questions -> {out_path}")


In [ ]:
# Preview one generated item, formatted as a quiz question
sample_files = sorted(questions_dir.glob("question_*.json"))
with open(sample_files[0], encoding="utf-8") as f:
    sample = json.load(f)

q = sample["questions"][0]
print(f"Source: {sample['source_article']}\n")
print(q["question"], "\n")
for i, option in enumerate(q["options"]):
    marker = "->" if i == q["correct_index"] else "  "
    print(f" {marker} {chr(65 + i)}. {option}")
print(f"\nEvidence: \"{q['evidence_excerpt']}\"")


## Takeaways

- LLMs make competent item writers **when the output is constrained**
  (strict JSON) **and verified** (the verbatim-evidence check). Watch how
  many items the groundedness gate drops — that number *is* the hallucination
  rate of your test generator.
- The generated questions inherit the article's freshness: they are about
  events from today, which is what makes the Notebook 4 audit meaningful.

**Next:** Notebook 4 quizzes a *fresh, context-free* model on these
questions to measure genuine real-time awareness.
